<a href="https://colab.research.google.com/github/dinukadilshan03/RoadLens/blob/feature%2Fc1-base-model-training/ml/c1/notebooks/train_base_model_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Base Model Training Pipeline (B0 Baseline)

This notebook trains the **Base Model (B0 Baseline)** for **Component 1 (C1: Road Damage Detection & Temporal Consolidation)** using a YOLOv8m detector.

### Pipeline Configuration
- **Architecture:** `YOLOv8m` (25.9M parameters)
- **Dataset Config:** `ml/c1/configs/base_model_b0.yaml` (4 core distress classes: `D00`, `D10`, `D20`, `D40`)
- **Hyperparameters:** `ml/c1/configs/hyp_base.yaml` (AdamW, cosine learning rate scheduler, road distress augmentations)
- **Resolution:** `800x800` (preserves fine sub-surface crack features)
- **Primary Checkpoint:** `base_model_b0.pt`

### 1. Hardware & Compute Environment Verification

Queries the NVIDIA system management interface (`nvidia-smi`) to inspect available GPU hardware, driver version, CUDA capability, and total video memory (VRAM). Ensures the PyTorch CUDA runtime detects an active accelerator before initializing model weights.

In [3]:
!nvidia-smi

Sun Oct 18 08:34:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### 2. Persistent Storage Mount (Google Drive)

Mounts Google Drive to `/content/drive` to establish persistent storage. Because Colab execution environments are ephemeral and destroyed upon disconnection, checkpoints (`base_model_b0.pt`), loss curves, and evaluation metrics are copied to Drive to ensure zero data loss.

In [4]:
import os
from google.colab import drive

drive_mount_point = '/content/drive'
if not os.path.exists(drive_mount_point):
    drive.mount(drive_mount_point)
    print("Google Drive mounted successfully.")
else:
    print(f"Google Drive already mounted at {drive_mount_point}.")

Google Drive already mounted at /content/drive.


### 3. Repository Provisioning & Environment Setup

Clones or synchronizes the `feature/c1-base-model-training` branch from the RoadLens repository and configures the execution environment:
- **Idempotent Repository Management:** If `/content/RoadLens` already exists, it fetches and resets to the latest branch commit instead of failing with directory collision errors.
- **Dependency Installation:** Installs `ultralytics` and `pyyaml` without forcing upgrades on existing system libraries (`pandas`, `google-colab`), avoiding environment dependency conflicts.

In [5]:
import os
from pathlib import Path

repo_dir = Path('/content/RoadLens')
if not repo_dir.exists():
    !git clone -b feature/c1-base-model-training https://github.com/dinukadilshan03/RoadLens.git /content/RoadLens
else:
    print("Repository directory exists. Synchronizing with latest origin/feature/c1-base-model-training...")
    !git -C /content/RoadLens fetch origin feature/c1-base-model-training
    !git -C /content/RoadLens checkout feature/c1-base-model-training
    !git -C /content/RoadLens reset --hard origin/feature/c1-base-model-training

# Set active working directory to repository root
%cd /content/RoadLens

# Install YOLO framework and YAML parser while preserving Colab system dependencies
!pip install --quiet "ultralytics>=8.3.0" pyyaml

Cloning into '/content/RoadLens'...
remote: Enumerating objects: 349, done.
remote: Counting objects: 100% (349/349), done.
remote: Compressing objects: 100% (255/255), done.
remote: Total 349 (delta 121), reused 269 (delta 69), pack-reused 0 (from 0)
Receiving objects: 100% (349/349), 2.26 MiB | 10.90 MiB/s, done.
Resolving deltas: 100% (121/121), done.
/content/RoadLens
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 13.2 MB/s eta 0:00:00


### 4. Dataset Extraction & Verification

Prepares the 4-class road distress dataset for model training:
- **Exact Archive Path:** Loads `rdd_cleaned.zip` directly from Google Drive at `/content/drive/MyDrive/data/rdd_cleaned.zip`.
- **Extraction Guard:** Inspects `ml/c1/data/rdd_cleaned/train/images` before unzipping. If already extracted, redundant unzipping is bypassed.
- **Integrity Verification:** Asserts that `train`, `val`, and `test` splits contain valid image files and logs split counts.

In [6]:
import os
import shutil
from pathlib import Path

target_data_dir = Path('/content/RoadLens/ml/c1/data/rdd_cleaned')
target_train_dir = target_data_dir / 'train' / 'images'

# Check if dataset is already extracted
if target_train_dir.exists() and any(target_train_dir.iterdir()):
    train_count = len(list(target_train_dir.glob('*.jpg')))
    print(f"Dataset already extracted at {target_data_dir} ({train_count} training images found). Skipping extraction.")
else:
    # Exact path to dataset archive in Google Drive
    zip_path = Path('/content/drive/MyDrive/data/rdd_cleaned.zip')

    if not zip_path.is_file():
        raise FileNotFoundError(
            f"Dataset archive not found at exact path: {zip_path}. "
            "Please ensure 'rdd_cleaned.zip' exists inside your Google Drive 'data' folder."
        )

    print(f"Dataset archive detected: {zip_path}")
    print("Extracting dataset to /content/RoadLens/ml/c1/data/ (this may take ~1 minute)...")
    !unzip -q "{zip_path}" -d /content/RoadLens/ml/c1/data/
    print("Extraction complete.")

# Validate dataset split integrity
train_imgs = len(list((target_data_dir / 'train' / 'images').glob('*.*')))
val_imgs = len(list((target_data_dir / 'val' / 'images').glob('*.*')))
test_imgs = len(list((target_data_dir / 'test' / 'images').glob('*.*')))

print("\nDataset Split Verification Summary:")
print(f"  - Train: {train_imgs} images")
print(f"  - Val:   {val_imgs} images")
print(f"  - Test:  {test_imgs} images")
assert train_imgs > 0, "Train split is empty."
assert val_imgs > 0, "Val split is empty."


Dataset archive detected: /content/drive/MyDrive/data/rdd_cleaned.zip
Extracting dataset to /content/RoadLens/ml/c1/data/ (this may take ~1 minute)...
Extraction complete.

Dataset Split Verification Summary:
  - Train: 26869 images
  - Val:   5758 images
  - Test:  5758 images


In [9]:
import yaml
from pathlib import Path

config_path = Path('/content/RoadLens/ml/c1/configs/base_model_b0.yaml')

# Read the configuration file
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Original config:")
for k, v in config.items():
    if k in ['path', 'train', 'val', 'test']:
        print(f"  {k}: {v}")

# Update path values to point to the correct extracted dataset directory
config['path'] = '/content/RoadLens/ml/c1/data/rdd_cleaned'
config['train'] = 'train/images'
config['val'] = 'val/images'
config['test'] = 'test/images'

# Write back the updated configuration file
with open(config_path, 'w') as f:
    yaml.safe_dump(config, f, default_flow_style=False)

print("\nUpdated config successfully written:")
with open(config_path, 'r') as f:
    updated_config = yaml.safe_load(f)
    for k, v in updated_config.items():
        if k in ['path', 'train', 'val', 'test']:
            print(f"  {k}: {v}")

Original config:
  path: ../data/rdd_cleaned
  train: train/images
  val: val/images
  test: test/images

Updated config successfully written:
  path: /content/RoadLens/ml/c1/data/rdd_cleaned
  test: test/images
  train: train/images
  val: val/images


### 5. Base Model (B0) Training Execution

Executes `ml/c1/scripts/base_model_training.py` with the following configuration:
- `--data ml/c1/configs/base_model_b0.yaml`: Dataset configuration mapping class indices `0: Longitudinal_Crack_D00`, `1: Transverse_Crack_D10`, `2: Alligator_Crack_D20`, `3: Pothole_D40`.
- `--hyp ml/c1/configs/hyp_base.yaml`: Hyperparameter definitions including AdamW optimizer, cosine learning rate scheduler, box gain `7.5`, and pavement-aware augmentations.
- `--model yolov8m.pt`: YOLOv8 medium architecture pretrained on COCO.
- `--epochs 50`: Total epoch schedule with early stopping patience of 15 epochs.
- `--imgsz 800`: 800x800 input resolution to maintain crack feature map fidelity.
- `--batch 16`: Batch size balanced for VRAM headroom and batch-norm stability.
- `--device 0`: Targets primary CUDA accelerator.

In [10]:
!python ml/c1/scripts/base_model_training.py \
    --data ml/c1/configs/base_model_b0.yaml \
    --hyp ml/c1/configs/hyp_base.yaml \
    --model yolov8m.pt \
    --name base_model_b0 \
    --epochs 50 \
    --imgsz 800 \
    --batch 16 \
    --device 0

2026-10-18 08:41:25 [INFO] === Launching C1 Base Model Training ===
2026-10-18 08:41:25 [INFO] Dataset Config : /content/RoadLens/ml/c1/configs/base_model_b0.yaml
2026-10-18 08:41:25 [INFO] Hyperparameters: /content/RoadLens/ml/c1/configs/hyp_base.yaml
2026-10-18 08:41:25 [INFO] Base Weights   : yolov8m.pt
2026-10-18 08:41:25 [INFO] Epochs         : 50 (Early Stopping Patience: 15)
2026-10-18 08:41:25 [INFO] Resolution     : 800x800 | Batch Size: 16
2026-10-18 08:41:25 [INFO] Device         : 0 | Seed: 42 | Deterministic: True
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=/content/RoadLens/ml/c1/configs/hyp_base.yaml, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, dat

### 6. Checkpoint Archival & Metrics Export

Manages post-training persistence to Google Drive:
- Copies the best model weights (`base_model_b0.pt`) to `/content/drive/MyDrive/RoadLens_C1_Weights/`.
- Archives training diagnostics (`ml/c1/runs/train/base_model_b0/`) including precision-recall curves (`PR_curve.png`), F1 curves (`F1_curve.png`), confusion matrices, and `results.csv` into a compressed `.tar.gz` archive.

In [15]:
import os
import shutil
from pathlib import Path

drive_weights_dir = Path("/content/drive/MyDrive/RoadLens_C1_Weights")
drive_weights_dir.mkdir(parents=True, exist_ok=True)

# 1. Locate the best model checkpoint
checkpoint_candidates = [
    Path("/content/RoadLens/runs/detect/ml/c1/runs/train/base_model_b0-2/weights/best.pt"),
    Path("/content/RoadLens/ml/c1/weights/base_model_b0.pt"),
    *sorted(Path("/content/RoadLens").glob("**/weights/best.pt"), key=os.path.getmtime, reverse=True)
]
best_weights = next((p for p in checkpoint_candidates if p.is_file()), None)

if best_weights:
    # Also copy to local repo weights directory for consistency
    local_weights = Path("/content/RoadLens/ml/c1/weights/base_model_b0.pt")
    local_weights.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best_weights, local_weights)
    shutil.copy2(best_weights, local_weights.parent / "b0.pt")

    # Backup best checkpoint to Google Drive
    dest_weights = drive_weights_dir / "base_model_b0.pt"
    shutil.copy2(best_weights, dest_weights)
    print(f"SUCCESS: Best model checkpoint saved to Drive: {dest_weights} ({best_weights.stat().st_size / (1024*1024):.1f} MB)")
else:
    print("ERROR: Checkpoint best.pt could not be found.")

# 2. Archive complete training run plots and logs to Drive
run_dir_candidates = [
    Path("/content/RoadLens/runs/detect/ml/c1/runs/train/base_model_b0-2"),
    *sorted(Path("/content/RoadLens").glob("**/base_model_b0*"), key=os.path.getmtime, reverse=True)
]
run_dir = next((p for p in run_dir_candidates if p.is_dir() and (p / "results.csv").is_file()), None)

if run_dir:
    archive_dest = drive_weights_dir / "training_run_logs.tar.gz"
    !tar -czf "{archive_dest}" -C "{run_dir.parent}" "{run_dir.name}"
    print(f"SUCCESS: Training logs, confusion matrix, and curves backed up to Drive: {archive_dest}")
else:
    print("WARNING: Training run directory not found for archiving.")

SUCCESS: Best model checkpoint saved to Drive: /content/drive/MyDrive/RoadLens_C1_Weights/base_model_b0.pt (49.7 MB)
SUCCESS: Training logs, confusion matrix, and curves backed up to Drive: /content/drive/MyDrive/RoadLens_C1_Weights/training_run_logs.tar.gz
